# PanTS Dataset
## Exploratory Data Analysis
### Things to note:
- I dont possess the Test set. It is reserved for third-party evaluation by the developing team.
- The dataset is divided into chunks. chunk 1 → 00000001–00001000, chunk 2 → 00001001–00002000, ...
  - I have only downloaded the first chunk for now.
- `metadata.xlsx` does not contain the full labels. The complete metadata is only accessible via the test set.
- `metadata.xlsx` TR has the following labels: (PanTS ID, shape, spacing, ct phase,	sex,	age,	manufacturer,	manufacturer model,	study type,	site,	site detail,	site nationality,	study year,	tumor?,	structured report)
- `ImageTr` contains the actual CT scans
  - uses `.nii.gz` file format
  - 0001 - 1000 files
- `LabelTr` contains the annotations for those CT scans (with expert-validated annotations)
  - uses `.nii.gz` file format
  - 0001 - 9901 files

In [19]:
import os
import numpy as np
import nibabel
import monai
from monai.transforms import LoadImage

In [2]:
loader = LoadImage(image_only=True)

data_root = "/Volumes/BackupDrive/pants/data"
label_root = f"{data_root}/LabelTr"
ct_root = f"{data_root}/ImageTr"

case = "PanTS_00000001"
segmentations = f"{label_root}/{case}/segmentations"

print("Segmentation masks ---->")
for f in sorted(os.listdir(segmentations)):
    print(f)

num_masks = sorted(os.listdir(segmentations))
print(f"Number of masks: {len(num_masks)}")

Segmentation masks ---->
adrenal_gland_left.nii.gz
adrenal_gland_right.nii.gz
aorta.nii.gz
bladder.nii.gz
celiac_artery.nii.gz
colon.nii.gz
common_bile_duct.nii.gz
duodenum.nii.gz
femur_left.nii.gz
femur_right.nii.gz
gall_bladder.nii.gz
kidney_left.nii.gz
kidney_right.nii.gz
liver.nii.gz
lung_left.nii.gz
lung_right.nii.gz
pancreas.nii.gz
pancreas_body.nii.gz
pancreas_head.nii.gz
pancreas_tail.nii.gz
pancreatic_duct.nii.gz
pancreatic_lesion.nii.gz
postcava.nii.gz
prostate.nii.gz
spleen.nii.gz
stomach.nii.gz
superior_mesenteric_artery.nii.gz
veins.nii.gz
Number of masks: 28


In [3]:
combined_labels = loader(f"{label_root}/{case}/combined_labels.nii.gz")

print("Combined labels ---->")
print(f"Shape: {tuple(combined_labels.shape)}")
print(f"Voxel dimensions: {combined_labels.meta['pixdim'][1:4]}")
print(f"Unique label values: {np.unique(combined_labels.numpy())}")

Combined labels ---->
Shape: (512, 333, 200)
Voxel dimensions: [0.625 0.625 0.8  ]
Unique label values: [ 0.  1.  2.  3.  5.  6.  7.  8. 11. 12. 13. 14. 15. 16. 17. 18. 19. 20.
 21. 22. 24. 25. 26. 27.]


# Notes:
- `Voxel dimensions: [0.625 0.625 0.8  ]` -> spacing is different in z
- TODO: might need to resample all volumnes to isotropic spacing before training `Voxel dimensions: [0.625 0.625 0.8  ]`

In [4]:
lesion = loader(f"{segmentations}/pancreatic_lesion.nii.gz")

print(f"Lesion shape: {tuple(lesion.shape)}")
print(f"Lesion unique values: {np.unique(lesion)}")
print(f"Voxels: {np.count_nonzero(lesion)}")

Lesion shape: (512, 333, 200)
Lesion unique values: [0.]
Voxels: 0


In [5]:
vascular_masks = [
    "superior_mesenteric_artery.nii.gz",
    "celiac_artery.nii.gz",
    "veins.nii.gz",
    "postcava.nii.gz",
    "aorta.nii.gz"
]

for mask_name in vascular_masks:
    path = f"{segmentations}/{mask_name}"
    img = loader(path)
    data = img.numpy()
    print(f"\n=== {mask_name} ===")
    print(f"Shape: {data.shape}")
    print(f"Unique vals: {np.unique(data)}")
    print(f"Voxels: {np.count_nonzero(data)}")


=== superior_mesenteric_artery.nii.gz ===
Shape: (512, 333, 200)
Unique vals: [0. 1.]
Voxels: 3349

=== celiac_artery.nii.gz ===
Shape: (512, 333, 200)
Unique vals: [0. 1.]
Voxels: 1356

=== veins.nii.gz ===
Shape: (512, 333, 200)
Unique vals: [0. 1.]
Voxels: 13537

=== postcava.nii.gz ===
Shape: (512, 333, 200)
Unique vals: [0. 1.]
Voxels: 52615

=== aorta.nii.gz ===
Shape: (512, 333, 200)
Unique vals: [0. 1.]
Voxels: 156992


# Notes:
- All 5 masks are the same shape.
- All masks are clean binary with only 0 and 1 and there is no partial values, no noise.
- TODO: Use these masks to compute NCCN based circumferential contact angle

In [6]:
cases = sorted(os.listdir(label_root))
lesion_counts = {}

for case in cases:
    lesion_path = f"{label_root}/{case}/segmentations/pancreatic_lesion.nii.gz"
    if os.path.exists(lesion_path):
        data = loader(lesion_path).numpy()
        lesion_counts[case] = np.count_nonzero(data)

positive_cases = {k: v for k, v in lesion_counts.items() if v > 0}
print(f"Total cases: {len(cases)}")
print(f"Tumour-positive: {len(positive_cases)}")
print(f"Tumour-negative: {len(cases) - len(positive_cases)}")

Total cases: 9901
Tumour-positive: 1033
Tumour-negative: 8868


In [7]:
# Among positive cases, which have vascular mask overlap with lesion?
vascular_masks = [
    "superior_mesenteric_artery.nii.gz",
    "celiac_artery.nii.gz",
    "veins.nii.gz",
    "postcava.nii.gz",
    "aorta.nii.gz"
]

contact_cases = []

for case in positive_cases:
    seg_path = f"{label_root}/{case}/segmentations"
    lesion_bool = loader(f"{seg_path}/pancreatic_lesion.nii.gz").numpy() > 0

    for vessel in vascular_masks:
        vessel_path = f"{seg_path}/{vessel}"
        if os.path.exists(vessel_path):
            vessel_data = loader(vessel_path).numpy() > 0
            if np.any(lesion_bool & vessel_data):
                contact_cases.append({"case": case, "vessel": vessel})
                break

print(f"Positive cases with any vascular contact: {len(contact_cases)}")
print(f"Positive cases without vascular contact: {len(positive_cases) - len(contact_cases)}")

Positive cases with any vascular contact: 186
Positive cases without vascular contact: 847


# Notes:
- 847 (~8.6%)   —> Tumour + No vascular contact
- 186 (~1.9%)   —> Tumour + Vascular contact
- 8868 (~89.6%) -> No Tumour

- Should it be a binary resectability framing?
  - **Class 0**: No vascular involvement
  - **Class 1**: Any vascular contact


In [8]:
# Get first tumour-positive case
positive_case = list(positive_cases.keys())[0]
seg_path = f"{label_root}/{positive_case}/segmentations"

print(f"Case: {positive_case} [POSITIVE]")

# Lesion mask
lesion = loader(f"{seg_path}/pancreatic_lesion.nii.gz").numpy()
print(f"\n=== Lesion ===")
print(f"Shape:         {tuple(lesion.shape)}")
print(f"Unique vals:   {np.unique(lesion)}")
print(f"Non-zero:      {np.count_nonzero(lesion)}")
print(f"Tumour present: {np.count_nonzero(lesion) > 0}")

# Vascular masks
vascular_masks = [
    "superior_mesenteric_artery.nii.gz",
    "celiac_artery.nii.gz",
    "veins.nii.gz",
    "postcava.nii.gz",
    "aorta.nii.gz"
]

lesion_bool = lesion > 0

for mask_name in vascular_masks:
    vessel = loader(f"{seg_path}/{mask_name}").numpy()
    vessel_bool = vessel > 0
    overlap = np.any(lesion_bool & vessel_bool)
    print(f"\n=== {mask_name} ===")
    print(f"Non-zero:       {np.count_nonzero(vessel)}")
    print(f"Vessel present: {np.count_nonzero(vessel) > 0}")
    print(f"Lesion contact: {overlap}")

Case: PanTS_00000003 [POSITIVE]

=== Lesion ===
Shape:         (495, 349, 40)
Unique vals:   [0. 1.]
Non-zero:      1055
Tumour present: True

=== superior_mesenteric_artery.nii.gz ===
Non-zero:       487
Vessel present: True
Lesion contact: False

=== celiac_artery.nii.gz ===
Non-zero:       843
Vessel present: True
Lesion contact: False

=== veins.nii.gz ===
Non-zero:       4866
Vessel present: True
Lesion contact: False

=== postcava.nii.gz ===
Non-zero:       16278
Vessel present: True
Lesion contact: False

=== aorta.nii.gz ===
Non-zero:       22067
Vessel present: True
Lesion contact: False


# Notes:
- Shape is different compared to PanTS_00000001.
- TODO: will require resampling to a consistent size and spacing before training

In [9]:
ct = loader(f"{ct_root}/{positive_case}/ct.nii.gz").numpy()
print(f"CT shape: {tuple(ct.shape)}")

CT shape: (495, 349, 40)


# Notes:
- CT and masks are same in shape

In [10]:
print(contact_cases[:3])

[{'case': 'PanTS_00000086', 'vessel': 'veins.nii.gz'}, {'case': 'PanTS_00000167', 'vessel': 'veins.nii.gz'}, {'case': 'PanTS_00000231', 'vessel': 'veins.nii.gz'}]


In [11]:
case = "PanTS_00000086"
seg_path = f"{label_root}/{case}/segmentations"

print(f"Case: {case} [POSITIVE]")

# Lesion mask
lesion = loader(f"{seg_path}/pancreatic_lesion.nii.gz").numpy()
print(f"\n=== Lesion ===")
print(f"Shape:          {tuple(lesion.shape)}")
print(f"Unique vals:    {np.unique(lesion)}")
print(f"Non-zero:       {np.count_nonzero(lesion)}")
print(f"Tumour present: {np.count_nonzero(lesion) > 0}")

# Vascular masks
lesion_bool = lesion > 0

for mask_name in vascular_masks:
    vessel = loader(f"{seg_path}/{mask_name}").numpy()
    vessel_bool = vessel > 0
    overlap = np.any(lesion_bool & vessel_bool)
    print(f"\n=== {mask_name} ===")
    print(f"Non-zero:       {np.count_nonzero(vessel)}")
    print(f"Vessel present: {np.count_nonzero(vessel) > 0}")
    print(f"Lesion contact: {overlap}")

Case: PanTS_00000086 [POSITIVE]

=== Lesion ===
Shape:          (455, 371, 75)
Unique vals:    [0. 1.]
Non-zero:       6556
Tumour present: True

=== superior_mesenteric_artery.nii.gz ===
Non-zero:       2355
Vessel present: True
Lesion contact: False

=== celiac_artery.nii.gz ===
Non-zero:       1637
Vessel present: True
Lesion contact: False

=== veins.nii.gz ===
Non-zero:       27273
Vessel present: True
Lesion contact: True

=== postcava.nii.gz ===
Non-zero:       61551
Vessel present: True
Lesion contact: False

=== aorta.nii.gz ===
Non-zero:       65521
Vessel present: True
Lesion contact: False


In [12]:
# check CT scans shape
shapes = []
for case in cases:
    ct_path = f"{ct_root}/{case}/ct.nii.gz"
    if os.path.exists(ct_path):
        img = loader(ct_path)
        shapes.append(tuple(img.shape))

shapes = np.array(shapes)
print(f"Min shape: {shapes.min(axis=0)}")
print(f"Max shape: {shapes.max(axis=0)}")
print(f"Mean shape: {shapes.mean(axis=0).astype(int)}")

Min shape: [43 57  8]
Max shape: [ 753  512 1060]
Mean shape: [437 343 183]


# NOTES:
- Z-axis (slices) is most extreme --> 8 to 1,060 (132x difference)
- Reflects multi-site acquisition
- TODO: Resample to fixed target shape and isotropic spacing